In [7]:
import sys
import pickle
import csv
import base64
import numpy as np
import torch
import json

from tqdm.auto import tqdm
from pathlib import Path
from transformers import LxmertTokenizer, LxmertForQuestionAnswering
from torch.utils.data import Dataset, DataLoader
from typing import Literal

def load_json(path: Path | str) -> dict:
    with Path(path).open() as f:
        return json.load(f)

csv.field_size_limit(sys.maxsize)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
FIELDNAMES = ["img_id", "img_h", "img_w", "objects_id", "objects_conf", "attrs_id", "attrs_conf", "num_boxes", "boxes", "features"]

def build_tsv_index(filepath: Path) -> dict:
    index = {}
    
    with open(filepath, 'rb') as f:
        offset = f.tell()
        line = f.readline()
        while line:
            img_id = line.split(b'\t')[0].decode('utf-8')
            index[img_id] = offset
            offset = f.tell()
            line = f.readline()
    
    return index


def load_feature_at(tsv_path: Path, offset: int):
    with open(tsv_path, 'rb') as f:
        f.seek(offset)
        line = f.readline().decode('utf-8')
    
    item = dict(zip(FIELDNAMES, line.strip().split('\t')))
    
    num_boxes = int(item["num_boxes"])
    feats = np.frombuffer(base64.b64decode(item["features"]), dtype=np.float32).reshape(num_boxes, 2048).copy()
    boxes = np.frombuffer(base64.b64decode(item["boxes"]), dtype=np.float32).reshape(num_boxes, 4).copy()
    h, w = int(item["img_h"]), int(item["img_w"])
    
    boxes[:, (0, 2)] /= w
    boxes[:, (1, 3)] /= h
    
    return feats, boxes


class GQALxmertDataset(Dataset):
    def __init__(self, questions_path: Path, tsv_path: Path, ans2label: dict[str, int]):
        self.tsv_path = Path(tsv_path)
        self.ans2label = ans2label
        self.questions = self._load_questions(questions_path)
        self.index = build_tsv_index(self.tsv_path)

    def _load_questions(self, questions_path):
        questions = load_json(questions_path)
        return [
            {
                "question_id": question_id,
                "question": q["question"],
                "image_id": q["imageId"],
                "answer": q["answer"].strip().lower()
            }
            for question_id, q in questions.items()
        ]

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        q = self.questions[idx]
        feats, boxes = load_feature_at(self.tsv_path, self.index[q["image_id"]])
        label = self.ans2label.get(q["answer"], -1)
        return {
            "question": q["question"],
            "question_id": q["question_id"],
            "image_id": q["image_id"],
            "visual_feats": torch.tensor(feats, dtype=torch.float32),
            "visual_pos": torch.tensor(boxes, dtype=torch.float32),
            "label": torch.tensor(label, dtype=torch.long),
            "is_binary": q["answer"] in {"yes", "no"}
        }


class VQALxmertDataset(Dataset):
    def __init__(self, questions_path: Path, tsv_path: Path, img_id_prefix: str):
        self.tsv_path = Path(tsv_path)
        self.img_id_prefix  = img_id_prefix
        self.questions = self._load_questions(questions_path)
        self.index = build_tsv_index(self.tsv_path)

    def _load_questions(self, questions_path):
        return [
            {
                "question_id": q["question_id"],
                "question": q["question"],
                "image_id": q["image_id"]
            }
            for q in load_json(questions_path)["questions"]
        ]

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        q = self.questions[idx]
        img_key = f"{self.img_id_prefix}{q['image_id']:012d}"
        feats, boxes = load_feature_at(self.tsv_path, self.index[img_key])
        
        return {
            "question":     q["question"],
            "question_id":  q["question_id"],
            "image_id":     q["image_id"],
            "visual_feats": torch.tensor(feats, dtype=torch.float32),
            "visual_pos":   torch.tensor(boxes, dtype=torch.float32)
        }

# LXMERT - VQA predictions

In [11]:
@torch.inference_mode()
def generate_lxmert_vqa_predictions(model, tokenizer, dataloader, idx2ans: dict[int, str], output_path: Path):    
    model.eval()
    model = model.to(DEVICE)

    predictions = []

    for batch in tqdm(dataloader, desc="Generating predictions"):
        questions = batch["question"]

        encoding = tokenizer(
            questions,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(DEVICE)

        feats = batch["visual_feats"].to(DEVICE)
        boxes = batch["visual_pos"].to(DEVICE)

        logits = model(
            input_ids=encoding["input_ids"],
            attention_mask=encoding["attention_mask"],
            visual_feats=feats,
            visual_pos=boxes
        ).question_answering_score

        preds = logits.argmax(dim=-1)

        for question_id, pred_idx in zip(batch['question_id'], preds):
            predictions.append({
                "question_id": int(question_id),
                "answer": idx2ans[pred_idx.item()]
            })

    with open(output_path, 'w') as f:
        json.dump(predictions, f)
    
    print(f"Saved {len(predictions)} predictions to {output_path}")


@torch.inference_mode()
def compute_lxmert_gqa_accuracy(model, tokenizer, dataloader):
    model.eval()
    model.to(DEVICE)

    binary_correct = 0
    binary_total = 0

    other_correct = 0
    other_total = 0

    total_correct = 0
    total = 0

    for batch in tqdm(dataloader):
        encoding = tokenizer(
            batch["question"],
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(DEVICE)

        labels = batch["label"].to(DEVICE)

        logits = model(
            input_ids=encoding["input_ids"],
            attention_mask=encoding["attention_mask"],
            visual_feats=batch["visual_feats"].to(DEVICE),
            visual_pos=batch["visual_pos"].to(DEVICE)
        ).question_answering_score

        preds = logits.argmax(dim=-1)

        correct_mask = preds == labels

        total_correct += correct_mask.sum().item()
        total += labels.size(0)

        is_binary = batch["is_binary"].bool().to(DEVICE)

        # binary
        if is_binary.any():

            binary_correct += (
                correct_mask[is_binary]
            ).sum().item()

            binary_total += is_binary.sum().item()

        # non-binary
        other_mask = ~is_binary

        if other_mask.any():

            other_correct += (
                correct_mask[other_mask]
            ).sum().item()

            other_total += other_mask.sum().item()

    overall_acc = total_correct / total

    binary_acc = (
        binary_correct / binary_total
        if binary_total > 0 else 0
    )

    other_acc = (
        other_correct / other_total
        if other_total > 0 else 0
    )

    print(f"Overall Accuracy : {overall_acc:.4f}")
    print(f"Binary Accuracy  : {binary_acc:.4f}")
    print(f"Other Accuracy   : {other_acc:.4f}")

    return {
        "overall": overall_acc,
        "binary": binary_acc,
        "other": other_acc
    }

In [8]:
def lxmert_vqa_predictions():
    tokenizer = LxmertTokenizer.from_pretrained("unc-nlp/lxmert-vqa-uncased")
    model = LxmertForQuestionAnswering.from_pretrained("unc-nlp/lxmert-vqa-uncased")    

    print('Answer vocab size', model.config.num_qa_labels)
    
    ans2idx = load_json(Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/trainval_ans2label.json'))
    idx2ans = {v: k for k, v in ans2idx.items()}

    assert model.config.num_qa_labels == len(idx2ans)

    dataset = VQALxmertDataset(
        Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Questions/v2_OpenEnded_mscoco_test2015_questions.json'),
        Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/test2015_obj36/test2015_obj36.tsv'),
        img_id_prefix="COCO_test2015_"
    )

    dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=2)
    
    generate_lxmert_vqa_predictions(model, tokenizer, dataloader, idx2ans, Path("lxmert_vqa.json"))
    
    print('LXMERT VQA predictions generated')


lxmert_vqa_predictions()

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/856M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/856M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/455 [00:00<?, ?it/s]

Answer vocab size 3129


Generating predictions:   0%|          | 0/3499 [00:00<?, ?it/s]

Saved 447793 predictions to lxmert_vqa.json
LXMERT VQA predictions generated


# LXMERT - GQA accuracy

In [12]:
def lxmert_gqa_accuracy():
    tokenizer = LxmertTokenizer.from_pretrained("unc-nlp/lxmert-gqa-uncased")
    model = LxmertForQuestionAnswering.from_pretrained("unc-nlp/lxmert-gqa-uncased")
    
    print('Answer vocab size', model.config.num_qa_labels)
    
    ans2label = load_json(Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/trainval_ans2label.json'))
    label2ans = {v: k for k, v in ans2label.items()}
    
    dataset = GQALxmertDataset(
        Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/GQA questions/GQA questions/testdev_balanced_questions.json'),
        Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/gqa_testdev_obj36/vg_gqa_imgfeat/gqa_testdev_obj36.tsv'),
        ans2label
    )

    dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=2)
    
    print('LXMERT - GQA accuracy', compute_lxmert_gqa_accuracy(model, tokenizer, dataloader))

lxmert_gqa_accuracy()

Loading weights:   0%|          | 0/455 [00:00<?, ?it/s]

Answer vocab size 1842


  0%|          | 0/99 [00:00<?, ?it/s]

Overall Accuracy : 0.5929
Binary Accuracy  : 0.7717
Other Accuracy   : 0.4925
LXMERT - GQA accuracy {'overall': 0.592940054062649, 'binary': 0.7717127071823204, 'other': 0.4924872718241649}
